In [ ]:
# Import all required libraries, set up required functions, and set directory paths

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_validate, cross_val_predict
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score

# Function for plotting predicted values against actual values for each lattice parameter (a, b, and c)
# def PlotPredictions(Y_test, Y_pred_df, modelName, displayStatistics=True, MSEVals=None, MAEVals=None, R2Vals=None, df=None, col=None, labels=None, displayOthers=True):
#     fig, axes = plt.subplots(1, 3, figsize=(18, 10))

#     Y_test_df = Y_test.reset_index(drop=True)

#     # Creates 3 subplots for each lattice parameter
#     for i, param in enumerate(Y_test_df.columns):
#         ax = axes[i]

#         #This plots each point colored based off its chosen label if X_test_full and labels is passed
#         if col is not None and labels is not None:            
#             X_test_full = df.loc[Y_test.index].reset_index(drop=True)
            
#             arr = np.asarray(X_test_full[col])
#             missing = [f for f in labels if f not in arr]
#             if missing:
#                 raise ValueError(f"The following labels are not found in column {col} of X_test_full: {missing}")

#             labelsList = list(labels)
#             cmap = plt.get_cmap('tab10')
#             colors = {label: cmap(i % cmap.N) for i, label in enumerate(labelsList)}
            
#             labelsList = ["Other"] + labelsList
#             colors["Other"] = "gray"
            
#             for label in labelsList:
#                 if label == 'Other':
#                     mask = ~np.isin(X_test_full[col], labels)
#                     if not displayOthers or not mask.any():
#                         continue
#                 else:      
#                     mask = X_test_full[col] == label
                    
#                 # Plot each actual value against its predicted value as individual points
#                 ax.scatter(
#                     Y_test_df.loc[mask, param],
#                     Y_pred_df.loc[mask, f"{param}_pred"],
#                     color = colors[label],
#                     label = label.title() if mask is not None else None,
#                     alpha = 0.7
#                 )

#                 # Create line of perfect fit where actual value = predicted value
#                 line = [Y_test_df[param].min(), Y_test_df[param].max()]
#                 ax.plot(line, line, 'k--')
            
#             if col == 'crystal_system':
#                 ax.legend(title='Crystal System', fontsize=18, title_fontsize=18)
#             else:
#                 ax.legend(title=col, fontsize=18, title_fontsize=18)
        
#         #This plots each point as the same color
#         else:
#             # Plot each actual value against its predicted value as individual points
#             ax.scatter(Y_test_df[param], Y_pred_df[f"{param}_pred"])

#             # Create line of perfect fit where actual value = predicted value
#             line = [Y_test_df[param].min(), Y_test_df[param].max()]
#             ax.plot(line, line, 'r--')

#         ax.set_xlabel("Actual Value", fontsize=18)
#         ax.tick_params(labelsize=18)
#         ax.set_ylabel("Predicted Value", fontsize=18)
#         ax.set_title(f"Lattice Parameter '{param}' (Å)", fontsize=20)
        
#         # Display MSE and R2 statistics for each lattice parameter if desired
#         if displayStatistics:
#             if MSEVals is not None and MAEVals is not None and R2Vals is not None:
#                 if df is not None:
#                     text = f"MSE={MSEVals[i]:.4f}\nMAE={MAEVals[i]:.4f}\nR²={R2Vals[i]:.4f}\nFull Dataset Size={len(df)}"
#                 else:
#                     text = f"MSE={MSEVals[i]:.4f}\nMAE={MAEVals[i]:.4f}\nR²={R2Vals[i]:.4f}"
                
#                 ax.text(
#                     0.05, 0.95,
#                     text,
#                     transform=ax.transAxes,
#                     verticalalignment='top',
#                     bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor="gray"),
#                     fontsize=14
#                 )
    
#     fig.suptitle(f"Predicted vs. Actual Lattice Parameter Using {modelName}", fontsize=22)
    
#     plt.tight_layout()
#     plt.show()
def PlotPredictions(
    Y_test, Y_pred_df, modelName,
    displayStatistics=True, MSEVals=None, MAEVals=None, R2Vals=None,
    df=None, col=None, labels=None, displayOthers=True
):
    fig, axes = plt.subplots(1, 3, figsize=(18, 10))
    Y_test_df = Y_test.reset_index(drop=True)

    for i, param in enumerate(Y_test_df.columns):
        ax = axes[i]

        if col is not None and labels is not None:
            X_test_full = df.loc[Y_test.index].reset_index(drop=True)
            arr = np.asarray(X_test_full[col])
            missing = [f for f in labels if f not in arr]
            if missing:
                raise ValueError(f"The following labels are not found in column {col} of X_test_full: {missing}")

            labelsList = list(labels)
            cmap = plt.get_cmap('tab10')
            colors = {label: cmap(idx % cmap.N) for idx, label in enumerate(labelsList)}
            labelsList = ["Other"] + labelsList
            colors["Other"] = "gray"

            for label in labelsList:
                if label == "Other":
                    mask = ~np.isin(X_test_full[col], labels)
                    if not displayOthers or not mask.any():
                        continue
                else:
                    mask = X_test_full[col] == label

                ax.scatter(
                    Y_test_df.loc[mask, param],
                    Y_pred_df.loc[mask, f"{param}_pred"],
                    color=colors[label],
                    label=label.title(),
                    alpha=0.7
                )

                line = [Y_test_df[param].min(), Y_test_df[param].max()]
                ax.plot(line, line, 'k--', linewidth=1)

            leg = ax.legend(
                title=col.title() if col != 'crystal_system' else 'Crystal System',
                prop={'size': 16, 'weight': 'bold'}
            )
            leg.get_title().set_fontsize(18)
            leg.get_title().set_fontweight('bold')

        else:
            ax.scatter(Y_test_df[param], Y_pred_df[f"{param}_pred"], alpha=0.7)
            line = [Y_test_df[param].min(), Y_test_df[param].max()]
            ax.plot(line, line, 'k--', linewidth=1)

        # Bold axis labels, title, ticks
        ax.set_xlabel("Actual Value", fontsize=20, fontweight='bold')
        ax.set_ylabel("Predicted Value", fontsize=20, fontweight='bold')
        ax.set_title(f"Lattice Parameter '{param}' (Å)", fontsize=22, fontweight='bold')
        ax.tick_params(labelsize=18)
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            tick.set_fontweight('bold')

        # Display statistics box (bold text inside)
        if displayStatistics and MSEVals is not None and MAEVals is not None and R2Vals is not None:
            stats = (
                f"MSE={MSEVals[i]:.4f}\n"
                f"MAE={MAEVals[i]:.4f}\n"
                f"R²={R2Vals[i]:.4f}"
            )
            if df is not None:
                stats += f"\nTest Size={918}"
            ax.text(
                0.05, 0.95, stats,
                transform=ax.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray'),
                fontsize=16, fontweight='bold'
            )

    fig.suptitle(
        f"Predicted vs. Actual Lattice Parameter Using\n{modelName}",
        fontsize=24, fontweight='bold'
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# Initialize dataset paths (Experimentlly Observed)
# MPDatasetFeaturized = "<local_path>/FuelComp/MachineLearning/Dataset/MP_Dataset_Featurized_Experimental.csv"
# TrainingDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/Training_Dataset_Experimental_CrossVal.csv"
# ModelMetricsDir = "<local_path>/FuelComp/MachineLearning/Dataset/ModelMetrics_Experimental_CrossVal.csv"
# PredictionDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/DatasetPredictions_Experimental_CrossVal.csv"

# # Initialize dataset paths (Theoretical Included)
MPDatasetFeaturized = "<local_path>/FuelComp/MachineLearning/Dataset/MP_Dataset_Featurized_Full.csv"
TrainingDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/Training_Dataset_Full_AvgOnly_CrossVal.csv"
ModelMetricsDir = "<local_path>/FuelComp/MachineLearning/Dataset/ModelMetrics_Full_AvgOnly_CrossVal.csv"
PredictionDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/DatasetPredictions_Full_AvgOnly_CrossVal.csv"

In [ ]:
# Read in featurized dataset and feature labels

df = pd.read_csv(MPDatasetFeaturized, low_memory=False)
display(df.head())
display(len(df))

featureLabels = joblib.load('FeatureLabels.joblib')
cs_featureLabels = joblib.load("cs_FeatureLabels.joblib")
sg_featureLabels = joblib.load("sg_FeatureLabels.joblib")

In [ ]:
# Filter out single element materials with identical reduced compositions and spacegroup numbers

df = df.sort_values("formation_energy_per_atom", ascending=True)

nelementsMask = df["nelements"] == 1
dupesMask = df.duplicated(subset=["composition_reduced", "spacegroup_num", "nsites"], keep="first")
mask = ~((nelementsMask) & (dupesMask))

df = df.loc[mask].reset_index(drop=True)

df = df.sort_values("composition_reduced", ascending=True).reset_index(drop=True)
df.groupby('nelements').count()

display(df.head())
display(len(df))

In [ ]:
# Filter out materials with exceptionally large lattice parameters a, b, and c

# Define maximum lattice parameter allowed
latParamThresh = 10 # angstroms

df = df[(df[['a','b','c']] <= latParamThresh).all(axis=1)]

display(df.head())
display(len(df))

In [ ]:
# Filter for materials contiaining a specific crystal system, if desired

filterCrystalSystem = False
csFilter = 'cubic'

if filterCrystalSystem:
    df = df[df['crystal_system'] == csFilter]

    display(df.head())
    display(len(df))

In [ ]:
# Use either space group OHE or crystal system OHE

useSpaceGroupOHE = False

if useSpaceGroupOHE:
    print('Using Spacegrup Number One Hot Encoding!')
    featureLabels = [col for col in featureLabels if col not in cs_featureLabels]
    featureLabels.remove('spacegroup_num')
else:
    print('Using Crystal System One Hot Encoding!')
    featureLabels = [col for col in featureLabels if col not in sg_featureLabels]

print(f'Chosen input features: {featureLabels}')

In [ ]:
# Filter out columns in featureLabels with no values (either all NaN, 0, or False)

emptyColsMask = ((df[featureLabels].isna()) | (df[featureLabels]==0)).all()

emptyCols = pd.Index(featureLabels)[emptyColsMask].tolist()
print(f"Dropped Empty Columns: {emptyCols}")

featureLabels = [col for col in featureLabels if col not in emptyCols]

df = df.drop(columns=emptyCols)

display(df)
print(featureLabels)

In [ ]:
# Specify feature labels to add/remove not originally indluced/removed during original datafeaturization (but feature is present in featurized dataframe)

# Removes any features with these substrings present in the feature label name
toRemove = ['minimum', 'maximum', 'range', 'mode', 'norm', 'max ionic'] # min, max, range, and mode statistics and norms hurt model predictions for impurities

# Adds feature labels with specified name(s)
toAdd = ['nsites']

filtered_featureLabels = [s for s in featureLabels if not any(sub in s for sub in toRemove)]

filtered_featureLabels.extend(toAdd)

print(filtered_featureLabels)

In [ ]:
# Anything with U > 0 goes to test set
test_mask = df['U'] > 0 

X_test = df.loc[test_mask, featureLabels]
Y_test = df.loc[test_mask, ['a', 'b', 'c']]

# Everything else goes to train set
X_train = df.loc[~test_mask, featureLabels]
Y_train = df.loc[~test_mask, ['a', 'b', 'c']]

mask_test_cubic = X_test['cs_cubic'] == True

print("Total dataset size:", len(df))
print("Training size:", len(X_train))
print("Test size:", len(X_test))
print("Cubic Only Test size:", len(X_test[mask_test_cubic]))

In [ ]:
# Create and train Dependent Random Forest Regressor model

# This model creates 1 random forest regressor where each tree in the forest predicts lattice parameters a, b, and c simultaneously
rf1Model = RandomForestRegressor(n_estimators=600, random_state=42, n_jobs=-1)

rf1Model.fit(X_train, Y_train)

In [ ]:
# Display statistics for Dependent Random Forest Regressor model

# Predict on test dataset
Y_pred_rf1 = rf1Model.predict(X_test)

# Print mean squared error, mean absolute error, and R2 values
MSErf1 = mean_squared_error(Y_test, Y_pred_rf1, multioutput='raw_values')
MAErf1 = mean_absolute_error(Y_test, Y_pred_rf1, multioutput='raw_values')
R2rf1 = r2_score(Y_test, Y_pred_rf1, multioutput='raw_values')
print("MSE:", MSErf1)
print("MAE:", MAErf1)
print("R²:", R2rf1)
MSErf1_cubic = mean_squared_error(Y_test[mask_test_cubic], Y_pred_rf1[mask_test_cubic], multioutput='raw_values')
MAErf1_cubic = mean_absolute_error(Y_test[mask_test_cubic], Y_pred_rf1[mask_test_cubic], multioutput='raw_values')
R2rf1_cubic = r2_score(Y_test[mask_test_cubic], Y_pred_rf1[mask_test_cubic], multioutput='raw_values')
print("MSE Cubic Only: ", MSErf1_cubic)
print("MAE Cubic Only: ", MAErf1_cubic)
print("R2 Cubic Only: ", R2rf1_cubic)

# Convert predicted values to dataframe
Y_pred_rf1 = pd.DataFrame(Y_pred_rf1, columns=['a_pred', 'b_pred', 'c_pred'])

# Plot predicted values vs. actual values
PlotPredictions(Y_test, Y_pred_rf1, "Lumped RF Regressor Model - U-Containing Compounds Only", True, MSErf1, MAErf1, R2rf1, df)
1
PlotPredictions(Y_test, Y_pred_rf1, "Lumped RF Regressor Model - U-Containing Compounds Only", True, MSErf1_cubic, MAErf1_cubic, R2rf1_cubic, df, 'crystal_system', ['cubic'], True)

In [ ]:
# Assuming you have predictions in Y_pred_df like this:
# Y_pred_df = pd.DataFrame({
#     'a_pred': model.predict(X_test)[:,0],
#     'b_pred': model.predict(X_test)[:,1],
#     'c_pred': model.predict(X_test)[:,2]
# })

# List of U-containing compounds of interest
u_compounds = [
    'UN', 'UC', 'UO2', 'U3O8', 'UO3', 'U4O9', 'U2N3', 'U2C3', 
    'USi2', 'U3Si2', 'UAl2', 'UAl3', 'UFe2'
]

# Filter only those that actually exist in the dataframe
existing_compounds = [c for c in u_compounds if c in df['formula_pretty'].unique()]
print(f"U-containing compounds in dataset: {existing_compounds}")

# Create mask: compounds in the list
UNUCUO2Mask = df['formula_pretty'].isin(existing_compounds)

# Apply mask to get test features and labels
X_test_masked = df.loc[UNUCUO2Mask, featureLabels]
Y_test_masked = df.loc[UNUCUO2Mask, ['a', 'b', 'c']]

# Display statistics for Dependent Random Forest Regressor model

# Predict on test dataset
Y_pred_rf1_masked = rf1Model.predict(X_test_masked)

# Convert predicted values to dataframe
Y_pred_rf1_masked = pd.DataFrame(Y_pred_rf1_masked, columns=['a_pred', 'b_pred', 'c_pred'])

def PlotPredictions_a_only_legend_fixed(Y_test, Y_pred_df, df, modelName):
    import matplotlib.pyplot as plt

    # Prepare test data
    X_test_full = df.loc[Y_test.index].reset_index(drop=True)
    Y_test_df = Y_test.reset_index(drop=True)

    # Initialize a wide plot
    fig, ax = plt.subplots(figsize=(12, 6))  # wider than tall

    # Labels to plot
    labels = existing_compounds

    # Use a colormap with enough distinct colors
    cmap = plt.get_cmap('tab20')  # 20 distinct colors
    colors = {label: cmap(idx % cmap.N) for idx, label in enumerate(labels)}

    # Plot each label
    for label in labels:
        mask = X_test_full['formula_pretty'] == label
        if mask.any():
            ax.scatter(
                Y_test_df.loc[mask, 'a'],
                Y_pred_df.loc[mask, 'a_pred'],
                color=colors[label],
                alpha=0.7,
                s=80,
                label=label
            )

    # 1:1 reference line
    min_a = Y_test_df['a'].min()
    max_a = Y_test_df['a'].max()
    ax.plot([min_a, max_a], [min_a, max_a], 'k--', linewidth=1)

    # Labels and title with bold formatting
    ax.set_xlabel("Actual 'a' (Å)", fontsize=20, fontweight='bold')
    ax.set_ylabel("Predicted 'a' (Å)", fontsize=20, fontweight='bold')
    ax.set_title(f"Predicted vs. Actual Lattice Parameter 'a' Using {modelName}", fontsize=22, fontweight='bold')

    # Legend: 2 columns
    leg = ax.legend(title='Compound', fontsize=16, ncol=2)
    leg.get_title().set_fontsize(18)
    leg.get_title().set_fontweight('bold')

    # Make legend item labels bold
    for text in leg.get_texts():
        text.set_fontweight('bold')

    # Bold ticks
    ax.tick_params(labelsize=16)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')

    plt.tight_layout()
    plt.show()

PlotPredictions_a_only_legend_fixed(Y_test_masked, Y_pred_rf1_masked, df, "Lumped RF Regressor Model")

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df_existing = df[df['formula_pretty'].isin(existing_compounds)]

display(df_existing)
